# Task 2 — 24类因子识别（多标签）baseline
IEEE BigData 2026 Explainable Suicide Risk Detection

DepRoBERTa + 24个sigmoid输出头 + BCE损失 + 逐类阈值校准 + Macro F1。
复用 common.py 地基。每帖判断 24 个因子各自有/无（multi-label）。

**运行前**：运行时类型选 GPU；确保 Drive 里有最新的 common.py + train_clean.csv。


## 0. 路径与超参


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/IEEE_BigData2026"   # <<<< 改成你的Drive文件夹
MODEL_TAG   = "deproberta"   # task2 baseline 用领域最强的 DepRoBERTa

MAX_LEN     = 512
BATCH_SIZE  = 8       # OOM 就降到 4
EPOCHS      = 5       # 多标签稀有类需要多几轮
LR          = 2e-5
SEED        = 42


## 1. 挂载 Drive + 依赖


In [ ]:
import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("非Colab")
assert os.path.isdir(PROJECT_DIR), f"找不到 {PROJECT_DIR}"
sys.path.insert(0, PROJECT_DIR); os.chdir(PROJECT_DIR)
print("工作目录:", os.getcwd(), "| 文件:", os.listdir(PROJECT_DIR))

import importlib, subprocess
def ensure(pkg, imp=None):
    try: importlib.import_module(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
for p,i in [("sentencepiece","sentencepiece"),("accelerate","accelerate"),("scikit-learn","sklearn")]:
    ensure(p,i)
import torch
assert torch.cuda.is_available(), "没GPU！运行时类型选GPU"
print("GPU:", torch.cuda.get_device_name(0))


## 2. 导入地基 + 数据


In [ ]:
import numpy as np, pandas as pd
from common import (load_data, get_fold, FACTORS, FACTOR_COLS,
                    macro_f1_multilabel, load_model_and_tokenizer, N_FOLDS)
from sklearn.metrics import f1_score

df = load_data()
print(f"数据 {len(df)} 行；因子数 {len(FACTORS)}")
# 24因子的0/1标签矩阵
Y = df[FACTOR_COLS].values.astype(np.float32)
print("各因子正样本数(前5和后5稀有的):")
counts = Y.sum(0).astype(int)
order = np.argsort(counts)
for i in list(order[:5]) + list(order[-5:]):
    print(f"  {FACTORS[i]:42s}: {counts[i]}")

def set_seed(s):
    import random; random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)


## 3. Dataset（多标签：labels是24维0/1向量）


In [ ]:
from torch.utils.data import Dataset
class FactorDataset(Dataset):
    def __init__(self, texts, labels, tok, max_len):
        self.texts=list(texts); self.labels=labels; self.tok=tok; self.max_len=max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc=self.tok(str(self.texts[i]), truncation=True, max_length=self.max_len,
                     padding="max_length", return_tensors="pt")
        item={k:v.squeeze(0) for k,v in enc.items()}
        if self.labels is not None:
            item["labels"]=torch.tensor(self.labels[i], dtype=torch.float)
        return item


## 4. 单fold训练+预测（multi_label模式）


In [ ]:
from transformers import Trainer, TrainingArguments

def train_one_fold(fold, df, Y, model_tag):
    tr_df, va_df = get_fold(df, fold)
    tr_idx = df.index[df["fold"]!=fold].tolist()
    va_idx = df.index[df["fold"]==fold].tolist()
    Ytr, Yva = Y[tr_idx], Y[va_idx]

    # 关键：problem_type="multi_label_classification" → 模型用 BCEWithLogitsLoss
    model, tok = load_model_and_tokenizer(model_tag, num_labels=len(FACTORS),
                                          problem_type="multi_label_classification")
    model.to("cuda")
    tr_ds=FactorDataset(tr_df["post"], Ytr, tok, MAX_LEN)
    va_ds=FactorDataset(va_df["post"], Yva, tok, MAX_LEN)

    use_bf16=torch.cuda.is_bf16_supported()
    args=TrainingArguments(
        output_dir=f"/content/ckpt_t2_{model_tag}_f{fold}",
        num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=32, learning_rate=LR,
        eval_strategy="no", save_strategy="no", logging_steps=50,
        warmup_ratio=0.1, weight_decay=0.01,
        bf16=use_bf16, fp16=not use_bf16, report_to="none", seed=SEED,
    )
    trainer=Trainer(model=model, args=args, train_dataset=tr_ds)
    trainer.train()
    # 预测概率（sigmoid）
    logits=trainer.predict(va_ds).predictions
    probs=torch.sigmoid(torch.tensor(logits)).numpy()
    del model, trainer; torch.cuda.empty_cache()
    return probs, va_idx


## 5. 跑满5fold，拼OOF概率


In [ ]:
oof_probs=np.zeros((len(df), len(FACTORS)), dtype=np.float32)
for fold in range(N_FOLDS):
    print(f"\n{'='*50}\n  Task2 Fold {fold} / {MODEL_TAG}\n{'='*50}")
    probs, va_idx = train_one_fold(fold, df, Y, MODEL_TAG)
    oof_probs[va_idx]=probs
    # 用0.5阈值快速看个中间分
    quick=f1_score(Y[va_idx], (probs>0.5).astype(int), average="macro", zero_division=0)
    print(f"  >> Fold {fold} Macro F1(阈值0.5) = {quick:.4f}")


## 6. 逐类阈值校准（task2提分关键：稀有类用更低阈值）


In [ ]:
# 默认0.5的整体Macro F1
f1_default = f1_score(Y, (oof_probs>0.5).astype(int), average="macro", zero_division=0)
print(f"★ 默认阈值0.5 整体 Macro F1 = {f1_default:.4f}\n")

# 逐类搜最优阈值（在OOF上为每个因子单独找让该类F1最大的阈值）
best_thresholds=np.full(len(FACTORS), 0.5)
for c in range(len(FACTORS)):
    best_t, best_f = 0.5, 0.0
    for t in np.arange(0.05, 0.95, 0.05):
        pred_c=(oof_probs[:,c]>t).astype(int)
        f=f1_score(Y[:,c], pred_c, zero_division=0)
        if f>best_f: best_f, best_t = f, t
    best_thresholds[c]=best_t

# 用校准后的逐类阈值
pred_calib=np.zeros_like(oof_probs, dtype=int)
for c in range(len(FACTORS)):
    pred_calib[:,c]=(oof_probs[:,c]>best_thresholds[c]).astype(int)
f1_calib=f1_score(Y, pred_calib, average="macro", zero_division=0)
print(f"★ 逐类阈值校准后 整体 Macro F1 = {f1_calib:.4f}")
print(f"  校准涨幅: {(f1_calib-f1_default)*100:+.2f} 个百分点\n")

# 每类的阈值和F1
print("各因子的最优阈值 + 该类F1:")
for c in np.argsort([f1_score(Y[:,c],pred_calib[:,c],zero_division=0) for c in range(len(FACTORS))]):
    fc=f1_score(Y[:,c], pred_calib[:,c], zero_division=0)
    print(f"  {FACTORS[c]:42s} 阈值={best_thresholds[c]:.2f}  F1={fc:.3f}  (正样本{int(Y[:,c].sum())})")


## 7. 保存OOF + 阈值 + 分数到 results/


In [ ]:
RESULTS_DIR=os.path.join(PROJECT_DIR,"results"); os.makedirs(RESULTS_DIR, exist_ok=True)
np.savez(os.path.join(RESULTS_DIR, f"oof_t2_{MODEL_TAG}.npz"),
         probs=oof_probs, thresholds=best_thresholds, row_ids=df["row_id"].values)
print(f"✓ OOF已存: results/oof_t2_{MODEL_TAG}.npz")

# 追加到 task2 分数表
sp=os.path.join(RESULTS_DIR,"scores_t2.csv")
row={"model":MODEL_TAG,"macro_f1_default":round(float(f1_default),4),
     "macro_f1_calibrated":round(float(f1_calib),4)}
if os.path.exists(sp):
    sdf=pd.read_csv(sp); sdf=sdf[sdf["model"]!=MODEL_TAG]
    sdf=pd.concat([sdf,pd.DataFrame([row])],ignore_index=True)
else: sdf=pd.DataFrame([row])
sdf.to_csv(sp,index=False)
print(f"✓ 分数已存: results/scores_t2.csv")
print(sdf.to_string(index=False))
print(f"\nTask2 baseline完成。Macro F1 = {f1_calib:.4f}（校准后）")
